In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
#from yahoofinancials import YahooFinancials
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.api as sm
from itertools import combinations
import requests
import datetime
import time
import warnings
warnings.filterwarnings('ignore')

# pwd

In [ ]:
STOCK_TICKERS = [
    "AAPL", "AMGN", "AMZN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS",
    "GS", "HD", "HON", "IBM", "JNJ", "JPM", "KO", "MCD", "MMM", "MRK",
    "MSFT", "NVDA", "NKE", "PG", "SHW", "TRV", "UNH", "V", "VZ", "WMT",
]

prices = yf.download(
    STOCK_TICKERS,
    start="2000-01-01",
    auto_adjust=True,
    progress=False,
    threads=False,
)["Close"]

if prices.empty:
    raise RuntimeError("Yahoo Finance returned no price data. Check your internet connection and ticker symbols.")

df1 = prices.dropna(axis=1, how="all").rename_axis("Date").reset_index()
if df1.shape[1] < 3:
    raise RuntimeError("Yahoo Finance returned data for fewer than two stocks.")

In [ ]:
df=df1
df.head()

In [ ]:
df.set_index(keys='Date',inplace=True)
df

## EDA

In [ ]:
df.dtypes

In [ ]:
pd.set_option('display.precision', 3)
df.describe()


In [ ]:
plt.figure(figsize=(20,20))
sns.heatmap(df.corr(method='pearson'),annot=True)

In [ ]:
df.isnull().sum()

In [ ]:
#df.fillna('bfill',inplace=True)
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df = df.apply(pd.to_numeric, errors='coerce')
df_lag = df.shift(1).copy()
df_lag.bfill(inplace=True)
data_log_returns = np.log(df / df.shift(1))

# Daily Linear Returns (%)
data_returns = df.pct_change(1).dropna()

# Removing Outliers beyond 3 standard deviations
data_returns = data_returns[data_returns.apply(lambda x: (x - x.mean()).abs() < (3 * x.std())).all(axis=1)]

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(data_returns)
rescaled_data = pd.DataFrame(scaler.fit_transform(data_returns),columns = data_returns.columns, index = data_returns.index)

In [ ]:
plt.figure(figsize=(20, 7))
'''
plt.figure(figsize=(30,20))
i=1
for c in rescaled_data.columns:
    plt.subplot(2,14,i)
    i+=1
    plt.plot(rescaled_data[c])
'''
plt.plot(rescaled_data['MMM'])
plt.title("MMM Return")
plt.ylabel("Return")
#plt.grid(True)
plt.show()

In [ ]:
X_train= rescaled_data[:int(len(rescaled_data) * 0.8)]
X_test= rescaled_data[int(len(rescaled_data) * 0.8):]
X_train_raw= data_returns[:int(len(rescaled_data) * 0.8)]
X_test_raw= data_returns[int(len(rescaled_data) * 0.8):]
#Use train_test_split instead with suffling off.
stock_tickers = rescaled_data.columns.values
n_tickers=len(stock_tickers)

In [ ]:
stock_tickers

In [ ]:
from sklearn.decomposition import PCA
pca=PCA()


In [ ]:
pca.fit_transform(X_train)

In [ ]:
pca.components_[0]

In [ ]:
pca.explained_variance_ratio_

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(14,4))
series1=pd.Series(pca.explained_variance_ratio_[:10]).sort_values()*100
series2=pd.Series(pca.explained_variance_ratio_[:10]).cumsum()*100

Cumm_Var=pd.DataFrame()
Cumm_Var['Explained Variance in %']=pd.DataFrame(series2)

series1.plot.barh(title='Explained Var Ratio by Top Factors',ax=axes[0])
series2.plot(ylim=(0,100),xlim=(0,9),title='Cumulative Explained Variance by factor',ax=axes[1])

Cumm_Var

In [ ]:
Cumm_Var=pd.DataFrame()
Cumm_Var['Explained Variance']=pd.DataFrame(series2)

In [ ]:
def PCWeights():
    weights= pd.DataFrame()

    for i in range(len(pca.components_)):
        weights['weights_{}'.format(i)]=pca.components_[i]/sum(pca.components_[i])
    weights = weights.values.T
    return weights

weights=PCWeights()

In [ ]:
weights

In [ ]:
pca.components_[0]

In [ ]:
top_Comp=5

topPortfolios = pd.DataFrame(pca.components_[:top_Comp], columns=df.columns)
eigen_portfolios = topPortfolios.div(topPortfolios.sum(1), axis=0)
eigen_portfolios.index = ['Portfolio {}'.format(i) for i in range(top_Comp)]
np.sqrt(pca.explained_variance_)
eigen_portfolios.T.plot.bar(subplots=True, layout=(int(top_Comp),1), figsize=(14,10), ylim= (-1,1))

In [ ]:
plt.figure(figsize=(20,11))
sns.heatmap(topPortfolios)

## Finding the best Eigen Portfolio

In [ ]:
def Sharpe_Ratio(port_returns, periods_per_yr=252):

    n_periods= periods_per_yr/port_returns.shape[0]
    annualized_return= np.power(np.prod(1+ port_returns),(n_periods))-1    # (1+r)^n -1
    annualized_vol= port_returns.std()*np.sqrt(periods_per_yr)
    annualized_sharpe= annualized_return/annualized_vol

    return annualized_return, annualized_vol, annualized_sharpe

In [ ]:
stock_tickers      #Declared  and Defined Above
n_tickers          #Declared and Defined Above

def OptimisedPortfolio():
    n_portfolios= len(pca.components_)
    annualized_ret = np.array([0.] * n_portfolios)
    sharpe_metric = np.array([0.] * n_portfolios)
    annualized_vol = np.array([0.] * n_portfolios)

    components= pca.components_

    max_sharpe=0
    results=pd.DataFrame()

    for i in range(n_portfolios):

        compo_weights= components[i]/sum(components[i])
        Eigen_port= pd.DataFrame(data ={'weightage': compo_weights*100}, index = stock_tickers)
        Eigen_port.sort_values(by=['weightage'],ascending=False,inplace =True)
        Eigen_port_returns = np.dot(X_train_raw.loc[:, Eigen_port.index], compo_weights)

        ar,vol,sharpe= Sharpe_Ratio(Eigen_port_returns)
        annualized_ret[i] = ar
        annualized_vol[i] = vol
        sharpe_metric[i] = sharpe
        sharpe_metric= np.nan_to_num(sharpe_metric)


    max_sharpe_index= np.argmax(sharpe_metric)
    print('Eigen portfolio with index %d with the highest Sharpe. Return %.2f%%, vol = %.2f%%, Sharpe = %.2f' %
         (max_sharpe_index,annualized_ret[max_sharpe_index]*100,
         annualized_vol[max_sharpe_index]*100, sharpe_metric[max_sharpe_index]))


    results=pd.DataFrame(data={'Return':annualized_ret,'Volatility':annualized_vol,'Sharpe Ratio': sharpe_metric})
    sharpe_values= results['Sharpe Ratio'].copy()
    results.sort_values(by=['Sharpe Ratio'], ascending=False, inplace=True)
    results.dropna(inplace=True)
    #print(results)
    return results, sharpe_values

In [ ]:
results, Sharpe_Values =OptimisedPortfolio()
results

In [ ]:
def compo_sharpe_graph(Sharpe_Values):
    fig, ax = plt.subplots()
    fig.set_size_inches(20, 4)
    ax.plot(Sharpe_Values, linewidth=3)
    ax.set_title('Sharpe ratio of eigen-portfolios')
    ax.set_ylabel('Sharpe ratio')
    ax.set_xlabel('Portfolios')

In [ ]:
compo_sharpe_graph(Sharpe_Values)

In [ ]:
weights = PCWeights()
portfolio = portfolio = pd.DataFrame()

def plotEigen(weights, plot=False, portfolio=portfolio):
    portfolio = pd.DataFrame(data ={'weights': weights.squeeze()*100}, index = stock_tickers)
    portfolio.sort_values(by=['weights'], ascending=False, inplace=True)
    print('Sum of weights of current eigen-portfolio: {}'.format(np.sum(portfolio)))
    portfolio.plot(title='Current Eigen-Portfolio Weights', figsize=(12,6), xticks=range(0, len(stock_tickers),1), rot=45,  linewidth=3)

    return portfolio

# Weights are stored in arrays, where 0 is the first PC's weights.
plotEigen(weights=weights[0])

## Backtesting

In [ ]:
def Backtest(eigen):
    best_sharpe=0
    best_return=0
    eigen_prtfi = pd.DataFrame(data ={'weights': eigen}, index = stock_tickers)
    eigen_prtfi.sort_values(by=['weights'], ascending=False, inplace=True)

    eigen_portfolio_returns = np.dot(X_test_raw.loc[:, eigen_prtfi.index], eigen)
    returns, vol, sharpe = Sharpe_Ratio(eigen_portfolio_returns)

    print('Current Eigen-Portfolio:\nReturn = %.2f%%\nVolatility = %.2f%%\nSharpe = %.2f' % (returns*100, vol*100, sharpe))


In [ ]:
#equal_weight_return=(X_test_raw * (1/len(pca.components_))).sum(axis=1)
#X_test_raw.sum(axis=1)
#np.cumprod(tt+1)

In [ ]:
def Backtest(eigen):

    Sharpe_Store=[]
    Return_Store=[]

    for i in range(0,len(pca.components_)):

        eigen_portfolio = pd.DataFrame(data ={'weights': eigen[i]}, index = stock_tickers)
        eigen_portfolio.sort_values(by=['weights'], ascending=False, inplace=True)

        eigen_portfolio_returns = np.dot(X_test_raw.loc[:, eigen_portfolio.index], eigen[i])
        eigen_portfolio_returns = pd.Series(eigen_portfolio_returns, index=X_test_raw.index)

        returns, vol, sharpe = Sharpe_Ratio(eigen_portfolio_returns)
        Sharpe_Store.append((sharpe,i))
        Return_Store.append((returns,i))
        #print(i)
    max_sharpe_ind= max(Sharpe_Store)[1]
    max_return_ind= max(Return_Store)[1]
    #print(max_sharpe_ind)
    #print(max_return_ind)


    #Equal Weighted Index
    equal_weight_return=(X_test_raw * (1/len(pca.components_))).sum(axis=1)     #Total Daily Return.
    returns_eq, vol_eq, sharpe_eq = Sharpe_Ratio(equal_weight_return)

    #Best Sharpe Ratio Portfolio
    best_sharpe_portfolio = pd.DataFrame(data ={'weights': eigen[max_sharpe_ind].squeeze()}, index = stock_tickers)
    best_sharpe_portfolio.sort_values(by=['weights'], ascending=False, inplace=True)

    best_sharpe_portfolio = np.dot(X_test_raw.loc[:, best_sharpe_portfolio.index], eigen[max_sharpe_ind])
    best_sharpe_portfolio = pd.Series(best_sharpe_portfolio, index=X_test_raw.index)
    returns_s, vol_s, sharpe_s = Sharpe_Ratio(best_sharpe_portfolio)

    #Best Return Portfolio
    best_return_portfolio = pd.DataFrame(data ={'weights': eigen[max_return_ind]}, index = stock_tickers)
    best_return_portfolio.sort_values(by=['weights'], ascending=False, inplace=True)

    best_return_portfolio = np.dot(X_test_raw.loc[:, best_return_portfolio.index], eigen[max_return_ind])
    best_return_portfolio = pd.Series(best_return_portfolio, index=X_test_raw.index)
    returns_r, vol_r, sharpe_r = Sharpe_Ratio(best_return_portfolio)

    #eigen_prti_returns = np.dot(X_test_raw.loc[:, eigen_prtfi.index], eigen)
    #eigen_portfolio_returns = pd.Series(eigen_prti_returns, index=X_test_raw.index)
    #returns, vol, sharpe = Sharpe_Ratio(eigen_portfolio_returns)

    print('Equal Weighted-Portfolio:\nReturn = %.2f%%\nVolatility = %.2f%%\nSharpe = %.2f\n\nCurrent Eigen-Portfolio with highest Sharpe Ratio:\nReturn = %.2f%%\nVolatility = %.2f%%\nSharpe = %.2f' % (returns_eq*100,vol_eq*100,sharpe_eq,returns_s*100, vol_s*100, sharpe_s))
    df_plot = pd.DataFrame({'EigenPorfolio Return': best_sharpe_portfolio, 'Equal Weight Index': equal_weight_return}, index=X_test.index)
    np.cumprod(df_plot+1).plot(title='Returns of the equal weighted index vs. eigen-portfolio',figsize=(12,6), linewidth=3)
    plt.show()

    print('Equal Weighted-Portfolio:\nReturn = %.2f%%\nVolatility = %.2f%%\nSharpe = %.2f\n\nCurrent Eigen-Portfolio with highest Return:\nReturn = %.2f%%\nVolatility = %.2f%%\nSharpe = %.2f' % (returns_eq*100,vol_eq*100,sharpe_eq,returns_r*100, vol_r*100, sharpe_r))
    df_plot = pd.DataFrame({'EigenPorfolio Return': best_return_portfolio, 'Equal Weight Index': equal_weight_return}, index=X_test.index)
    np.cumprod(df_plot+1).plot(title='Returns of the equal weighted index vs. eigen-portfolio',figsize=(12,6), linewidth=3)
    plt.show()

Backtest(weights)